# Single-location state and measurement optimization

$M=1$, $K\in\{3,4,5\}$, $N\in\{2,4,6,8,10\}$, with 50 instances per dimension, field family and $K$. The same field instances are used at every sensor budget. Field priors are uniform.

Dicke-sector weights and the POVM are optimized jointly by an exact SDP, subject to numerical feasibility and optimality checks. Reflection symmetry is not imposed. Product, GHZ and equal-sector states are unchanged; PGM uses the same weights as the optimal-POVM comparison.


## Environment

Install missing dependencies in the active kernel. Restart the kernel after installation if already loaded packages were upgraded.

In [ ]:
from pathlib import Path
import sys
import importlib.util
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT / "problem-3-single-location"
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Open from the project folder or repository root")

required = ("cvxpy", "numpy", "pandas", "scipy", "matplotlib")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print("Installing missing dependencies into the active kernel:", ", ".join(missing))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")
    ])
    importlib.invalidate_caches()
    raise RuntimeError("Dependencies installed. Restart the kernel, then run all cells.")
print("Python environment:", sys.executable)


In [ ]:
from IPython.display import display
import pandas as pd

sys.path.insert(0, str(ROOT))
KS = (3, 4, 5)
NS = (2, 4, 6, 8, 10)
RUN_ANALYSIS = False
SOLVER = "auto"
SDP_EPS = 1e-8


## Saved results

The runner validates the generated inputs in `inputs/1d/` and `inputs/2d/`. Results and resumable checkpoints are saved under `results/full/`. With `RUN_ANALYSIS=False`, only saved results are loaded.


In [ ]:
if RUN_ANALYSIS:
    from src.analysis import run_analysis
    status = display({"text/plain": "Starting analysis"}, raw=True, display_id=True)

    def show_progress(message):
        status.update({"text/plain": message}, raw=True)

    measurements = run_analysis(ROOT, ks=KS, ns=NS, progress=show_progress, solver=SOLVER, sdp_eps=SDP_EPS)
else:
    measurements = pd.read_csv(ROOT / "results/full/measurements.csv")
    print(f"Loaded {len(measurements)} saved state evaluations; M=1, K={KS}, N={NS}.")


## Results

Each figure retains three panels: Gaussian, sinusoidal and quadratic. Performance curves give 1D and 2D equal weight and average equally over $K=3,4,5$, using 300 instances per curve point. Figure 1 compares four states with optimal POVMs. Figure 2 shows paired excess error from PGM as grouped bars. Curves and bars show means; summary tables include stratified 95% confidence intervals. Separate tables retain each dimension and $K$.

Figure 3 pools the optimized weights from both dimensions and compares $p_w$ with $p_{N-w}$, using the existing sensor-budget colors. Central sectors are omitted. Reflection asymmetry is a diagnostic, not an optimization constraint.


In [ ]:
import importlib
import matplotlib.pyplot as plt
import src.reporting as reporting

reporting = importlib.reload(reporting)
results = reporting.make_results(measurements, ROOT)
for name in ("overall_errors", "overall_degradation", "errors", "degradation", "symmetry"):
    print({"overall_errors": "Optimal-POVM error across K", "overall_degradation": "PGM excess error across K", "errors": "Optimal-POVM error by K", "degradation": "PGM excess error",
           "symmetry": "Reflection asymmetry"}[name])
    table = results[name]
    if name != "symmetry":
        table = table.assign(estimate=table.apply(
            lambda r: f"{r['mean']:.6f} ± {r['ci95']:.6f}", axis=1))
        index = [col for col in ("dimension", "family", "k", "m", "n") if col in table.columns]
        table = table.pivot(index=index, columns="state", values="estimate")
    display(table)
for figure in results["figures"]:
    display(figure)
    plt.close(figure)


Figures are exported as PDF and PNG under `results/full/`, together with error summaries, sector weights, runtime and numerical validation tables.
